# 18BI · Week 2 — Production vs Analytical Systems

**Business Intelligence** · FNSPE CTU in Prague · winter semester 2026/27
Matej Mojzeš · [LinkedIn](https://www.linkedin.com/in/matejmojzes)

---

Last week we stopped at the course map. This week picks up from there — who does what in a
data team, which decisions BI serves, where AI really fits — and then the distinction
everything else rests on: the systems that *run* a business versus the systems that
*explain* it. We finish by taking last week's revenue question to a computer.

This notebook is the whole lecture. We go through it together in class, and it is written
so that you can also work through it on your own. The hands-on part has its own notebook:
[`exercises/02_first_queries.ipynb`](../exercises/02_first_queries.ipynb).

**Contents**

0. Setup
1. Where we left off
2. BI vs Data Engineering vs Data Science
3. From reporting to decision intelligence
4. Where AI actually sits in a BI stack
5. Production versus analytical systems
6. Facts, dimensions and grain
7. Your first queries
8. Key takeaways
9. Check yourself
10. Homework and *zápočet*

Appendix — the vocabulary

## 0 · Setup — first, while there is time to fix it

The second half of today is hands-on, so we start by making sure your machine is ready. If
you do not have the course repository yet, clone it from GitLab first. Then, from its root
folder:

```bash
git pull
uv sync
uv run jupyter lab
```

Open [`exercises/00_hello_world.ipynb`](../exercises/00_hello_world.ipynb) and run it top to
bottom. It checks that Jupyter runs on the project's Python and that the packages are there,
runs a first query, and ends with either `ALL GOOD — you are ready for week 2.` or a short
list of what is missing.

No `uv`? `python -m venv .venv && .venv/bin/pip install -r requirements.txt` works too.

**Not `ALL GOOD`?** In class: do not fight it now — sit next to someone whose environment
works, and we fix yours in the break. On your own: the notes at the bottom of `hello_world`
cover the usual problems, and if they do not help, email me the output.

## 1 · Where we left off

Four claims about what BI is:

1. separate the systems that *run* the business from the systems that *understand* it
2. a metric has *one* definition, and that definition is the real deliverable
3. history is a first-class citizen
4. the output is a changed decision, not a produced chart

And one experiment. Eight of you answered *"what was our revenue?"* from the same seven order
lines, and produced six different numbers, between 9 850 and 19 240:

![Every response, against the six rules](../files/week01_responses.png)

Only three of the eight numbers could be reproduced from the rule written beside them. The
full story is in [lecture 01, §5](01_bi_in_the_age_of_ai.ipynb). Keep it in mind — the second
half of today is about the half of that problem SQL can fix.

> **Where the term comes from.** It is older than you might expect — Hans Peter Luhn used it at IBM
> in 1958; Howard Dresner popularised the modern sense at Gartner around 1989. What changed
> in between is that companies started keeping more data than any person could read.

## 2 · BI vs Data Engineering vs Data Science

Same company, same data, four people, four questions:

| Role | The question they own | Typical output |
|---|---|---|
| **Data engineer** | *Is the data here, fresh, and correct?* | pipelines, infrastructure, SLAs |
| **Analytics / BI engineer** | *What happened — and do we all agree what "it" means?* | models, metric definitions, marts |
| **Data scientist** | *Why did it happen, and what happens next?* | forecasts, segmentations, experiments |
| **Decision intelligence** | *So what do we do — and did it work?* | decisions, and measurement of decisions |

These are four *questions*, not four job adverts. In a start-up one person does all four
before lunch; in a bank there are four departments that email each other.

**A career note, since you are the people who will have to choose.** Analytics
engineering — claim 2 above, done properly — is currently probably the most undersupplied of
these four, and it suits FNSPE graduates well: it rewards precision, modelling, and
stubbornness about definitions. LLMs already write much of its code, but their answers are
only as good as the definitions this role produces. Given an insurance company's raw database,
GPT-4 answered 16% of business questions correctly; given the business context too, 54%
([Sequeda et al., 2023](https://arxiv.org/abs/2311.07509)). A 2026 benchmark found that a short
hand-written document of definitions mattered more than which model was used
([Rumiantsau & Fokeev, 2026](https://arxiv.org/abs/2604.25149)) — both studies, it is worth
noting, by companies that sell tools for writing such definitions down.

### One question, four people

Take the CEO's question from last week — *"why did revenue drop 12% last week?"* — and
follow it through the four roles:

1. **The data engineer** checks the pipes. Did last week's orders load completely? Was
   there an outage on Thursday? Did a source system change its export format?
2. **The BI / analytics engineer** checks the definition. Is this "revenue" computed the
   same way as in the previous weeks' reports? Did refunds start being booked differently?
3. **The data scientist** asks why. Is the drop in one segment, one channel, one region?
   Is it seasonal? Is it within normal noise?
4. **Whoever owns the decision** asks *so what*. Run a promotion, fix the checkout, or wait
   a week and see?

The order matters, because each step depends on the one before it. There is no point
explaining a drop that is really a failed data load, and no point acting on an explanation
built on a changed definition. In a small company one person walks all four steps before
lunch; in a large one they are four departments. The sequence is the same.

## 3 · From reporting to decision intelligence

Not all decisions are the same shape, and this is *why* BI architectures look the way
they do. Three horizons, one company:

| | **Operational** | **Managerial** | **Strategic** |
|---|---|---|---|
| horizon | minutes – hours | weeks – a quarter | quarters – years |
| example | *this delivery is late — notify the customer?* | *is the Competitive segment growing fast enough to justify the campaign?* | *do we enter the Austrian market?* |
| how many | thousands per day | dozens per quarter | a handful per year |
| who decides | often the system itself | a manager, with a dashboard | the board, with a deck |
| needs | **low latency**, fine grain | **consistency**, comparability | **context**, external data |
| cost of being wrong | small, repeated | moderate | existential |

Read that table as an engineering specification, because that is what it is:

* operational decisions force you to care about **latency and freshness** → week 3, batch vs stream
* managerial decisions force you to care about **consistent definitions and history** → weeks 4 and 5
* strategic decisions force you to care about **external data and context** → weeks 10 and 11

"Decision intelligence" is just the name for taking that last column seriously: measuring
whether the decision was any good, and feeding that back.

### What analytics is asked for

A useful ladder, and roughly the order in which an organisation matures:

| | question | example | tools |
|---|---|---|---|
| **descriptive** | what happened? | revenue fell 12% last week | reports, dashboards |
| **diagnostic** | why did it happen? | one category, one region | OLAP, drill-down |
| **predictive** | what will happen? | next quarter's demand | statistics, ML |
| **prescriptive** | what should we do? | reprice, restock, call them | optimisation, decision support |

Most companies believe they need the bottom two and actually need the top two done
reliably. A trustworthy descriptive layer is worth more than an untrustworthy predictive
one, and it is also the foundation the predictive one needs.

## 4 · Where AI actually sits in a BI stack

This is most likely to be oversold to you by a vendor. Five honest places:

**1. Natural Language Query (NLQ)** — "show me revenue by region last quarter".
Works beautifully on a well-modelled warehouse with a semantic layer. Produces confident
nonsense on a pile of CSVs. *The quality of NLQ is a function of the modelling underneath it,
not of the model on top.* This is the single most important sentence in this section.

**2. Natural Language Generation (NLG)** — auto-written commentary on a dashboard.
Genuinely useful for "what changed since last week" summaries. Fails at *why*.

**3. Anomaly detection** — flagging the unusual. The hard part is not detection, it is the
false-positive flood: an alert that fires daily is an alert nobody reads.

**4. Insight discovery** — "here are seven interesting patterns in your data". The tool
slices the data every way it can and shows you the slices that stand out. The catch: check
enough slices and some will stand out by pure chance. At the usual 5% threshold, about one
random slice in twenty looks "significant" — so a tool that checks 300 combinations hands you
around fifteen discoveries from data that contains none, and does not tell you how many it
checked. Run on purely random numbers, exactly this produces findings like *"Czech customers
spend 15% more per order on GPS watches"*. Statisticians call this the **multiple comparisons
problem** and have known about it for a century; automation made it industrial. Treat an
automatic insight as a question, not an answer: does it hold next month?

**5. AI as a producer of pipeline code** — the quiet one, and the one that has actually
changed my working life. We will use it in this course, deliberately, and you are
encouraged to. But you own what you commit.

### The counterweight

> **An insight nobody can trace is worse than no insight.**

If a model tells your CFO that the Austrian market is underperforming, the first question
will be *"according to which definition, over what period, from what data?"* — and
"the AI said so" ends careers.

This is why weeks 12 and 13 exist (lineage, governance, data quality) and why they are
not the boring administrative part of the course. They are what makes the AI part usable
in a real organisation rather than a demo.

## 5 · Production versus analytical systems

The one-sentence version, which was claim 1 last week: the systems that run a business are
built to **record** things — safely, quickly, one at a time. The systems that explain it are
built to **read** enormous amounts of what was recorded. Different jobs, different shapes,
different machines.

### Two kinds of database

| | **operational (OLTP)** | **analytical (OLAP)** |
|---|---|---|
| typical question | *"what is in basket 88213?"* | *"how did baskets change last quarter?"* |
| rows touched | one, or a handful | millions |
| writes | constant, tiny | periodic, bulk |
| optimised for | latency and integrity | aggregation and scan speed |
| history | usually overwritten | deliberately kept |
| shape | normalised, many narrow tables | denormalised, few wide tables |
| if it goes down | the business stops | the reports are late |

Running analytics directly on the operational database is the first thing every company
tries and the first thing every company stops doing: the queries are slow, and while they
run they compete with paying customers.

### Three consequences

The split has three consequences, and they are what this week is really about.

**Latency — how fresh does it need to be?** Analytical data is always a copy, and a copy is
always behind. *How far* behind is a business decision, not a technical one, and it follows
straight from the decision horizons in §3: an operational decision may need minutes, a
managerial one is happy with last night, a strategic one with last month. Fresher always costs
more. Ask for "real time" only when somebody will act in real time.

**Historisation — remembering what the source forgets.** The e-shop overwrites a customer's
address when they move; the CRM overwrites their segment when it changes. For running the
business, that is correct. For analysing it, it is a disaster.

Say customer C-100 was *Casual* until March, then started buying seriously, and the CRM moved
them to *Competitive*. The CRM now knows only *Competitive*. Ask it for "revenue from
Competitive customers in January" and it will happily count C-100's January orders — made
when they were Casual. Last quarter's report, rerun today, gives a different number than it
did last quarter. Nobody changed the data; the past was simply overwritten.

The analytical side keeps the history the operational side throws away. How, exactly, is
week 4.

**Operational decisions — closing the loop.** Analytical results no longer stop at a
dashboard. Increasingly they flow *back* into the operational systems: a churn-risk score in
front of the call-centre agent, a fraud flag on a payment, a reorder suggestion in the
warehouse. At that point the analytical system has become part of production — and inherits
production's expectations about uptime and freshness.

### How data gets from one side to the other

Sources on the left, decisions on the right, and five layers in between. Every BI product
ever sold is a variation on this picture — and every week of this course is one box in it.

![The course map](../files/course_map.svg)

Read it as a sequence of increasingly opinionated steps:

1. **Extract** — copy data out of the source systems, changing nothing. Being able to prove
   what the source actually said is worth more than it sounds.
2. **Clean and conform** — types, names, duplicates, orders with no customer.
   Still no business opinions.
3. **Model** — facts, dimensions, grain. This is where the opinions start.
4. **Define** — metrics, once, in one place.
5. **Present** — reports, dashboards, OLAP, an LLM answering questions.

Notice that the *presentation* layer is last and thinnest. Most of BI happens before
anybody sees a chart. Steps 1 to 4 are next week's subject.

## 6 · Facts, dimensions and grain

Before we touch the data with SQL, three words you will use every week from now on.

Take the seven order lines from the handout. Which columns are measurements, and which
are context?

| column | fact or dimension? | why |
|---|---|---|
| `qty`, `unit price`, `discount`, `shipping` | **facts** | you add them up |
| `order`, `line` | fact keys | they identify the row, they are not measures |
| `product` | **dimension** | "revenue *by product*" |
| `date` | **dimension** | "revenue *by month*" |
| `customer` | **dimension** | "revenue *by segment*" |
| `status`, `VAT` | **dimension-ish** | you filter by them, but they are too small to deserve a table |

The test is the word **"by"**. Anything that can follow it is a dimension. Anything you
would `sum()` is a fact.

Then the grain — one sentence, written before any SQL:

> **One row per order line.**

Everything else follows from that. Shipping is charged per *order*, so at this grain it
has to be split across the lines, or you count it twice. That is precisely the mistake
behind 19 240 last week: 15 900, plus VAT.

### The star

![Star schema for the example](../files/star_schema.svg)

It is called a star because that is what it looks like, and it is the default for a
reason: any question of the form *"total X by Y and Z"* is one join away, and it is
comprehensible to people who do not write SQL.

## 7 · Your first queries

Last week's question, in SQL, with DuckDB, on the seven rows from the handout.

### The tools

Everything open source, everything on your own laptop, nothing expiring at the end of term.

| | | why this one |
|---|---|---|
| **DuckDB** | the analytical database | a real columnar OLAP engine in a single file. No server, no account, reads CSV and Parquet directly. |
| **dbt** | the transformation layer | turns SQL into a dependency graph with tests, documentation and lineage. |
| **Python** | extraction, enrichment, glue | the parts SQL is bad at. |
| **Jupyter** | how these materials are written | you are reading one. |
| **Streamlit** | dashboards, later in the term | reporting without a frontend detour. |

In industry you will meet the same boxes wearing other names:

| box | in this course | cloud platforms | Microsoft SQL Server (Enterprise) |
|---|---|---|---|
| extract & load | Python | Fivetran, Airbyte | Integration Services (SSIS) |
| warehouse | DuckDB | Snowflake, BigQuery, Databricks | the SQL Server database engine |
| transform | dbt | dbt, Matillion | SSIS packages, stored procedures |
| semantic layer & OLAP | dbt, later in the term | dbt Semantic Layer, Cube, Looker | Analysis Services (SSAS) |
| reports & dashboards | Streamlit | Tableau, Power BI | Power BI Report Server (SSRS up to 2022) |

The cloud column is a mix of vendors, one per box. The SQL Server column is a single product
family covering every box — the classic setup in many large companies. Real stacks mix the
columns, too: dbt runs on top of SQL Server, Power BI on top of Snowflake. The boxes are
identical. Learn the boxes.

### The exercise

> [`exercises/02_first_queries.ipynb`](../exercises/02_first_queries.ipynb)

Every query is a few lines, with hints and a solution folded away beneath it. Work through it
top to bottom: try each query first, open a hint when you are stuck, and the solution only to
check. You will:

* see grain in action: seven rows, but only six orders
* write *your own* rule from last week as a query, and check it against the number you submitted
* write three of the handout's rules, then meet the fan-out and fix it
* *(optional)* put the pieces together into one more rule

What to take away: **SQL fixes the arithmetic. It does not choose the rule.**


## 8 · Key takeaways

1. **Four questions, four roles** — is the data there, what does it mean, why did it happen,
   what do we do — and each depends on the one before.
2. **Decision horizons are an engineering specification.** Operational decisions need
   latency, managerial ones need consistency, strategic ones need context.
3. **AI in BI is only as good as the modelling underneath it**, and an insight nobody can
   trace is worse than no insight.
4. **Production systems record; analytical systems read.** Different jobs, different shapes,
   different machines.
5. **Analytical data is a copy**, so it is always behind (latency), it has to remember what
   the source forgets (history), and increasingly it feeds back into production.
6. **Facts are what you sum, dimensions are what you group by, and the grain is what one row
   means.** Get the grain wrong and you count things twice without noticing.
7. **SQL fixes the arithmetic. It does not choose the rule.**

## 9 · Check yourself

Try to answer before unfolding.

**1. The CEO asks why revenue fell. Who should look first — the data scientist or the data engineer?**

<details><summary>Answer</summary>

The data engineer. There is no point explaining a drop that turns out to be a failed data load.

</details>

**2. Your manager asks for a "real-time" sales dashboard. What do you ask back?**

<details><summary>Answer</summary>

Who will act on it, and how quickly? If the decisions it supports are made weekly, last night's data is enough — and far cheaper than real time.

</details>

**3. Why not simply run your analytics on the e-shop's own database?**

<details><summary>Answer</summary>

It is built for many tiny writes, not huge reads: analytical queries are slow there and compete with paying customers. And it overwrites history, so the past is not there to analyse.

</details>

**4. The CRM overwrites a customer's segment when it changes. What goes wrong in a "revenue by segment" report — and when would you notice?**

<details><summary>Answer</summary>

All of a customer's past revenue moves to their *current* segment, so last quarter's report gives a different answer every time it is rerun. You notice only if somebody compares against an old copy — usually much later.

</details>

**5. In the handout table, which columns are facts and which are dimensions? What is the grain?**

<details><summary>Answer</summary>

Facts: quantity, unit price, discount, shipping. Dimensions: product, date, customer, and — small enough to stay as plain columns — status. Grain: one row per order line.

</details>

**6. Why does summing the `shipping` column give the wrong total, and what is the general fix?**

<details><summary>Answer</summary>

Shipping belongs to the order, but the table repeats it on every line, so orders with several lines are counted several times — a fan-out. The fix: change the grain first (one row per order), then aggregate.

</details>

**7. An LLM answers "what was our revenue?" instantly and confidently. What would you want to know before trusting the number?**

<details><summary>Answer</summary>

Which definition it used (completed orders only? refunds? VAT? shipping?), on which data, for which period — in short, whether it went through an agreed definition or improvised one.

</details>

## 10 · Homework and *zápočet*

### How you get the *zápočet*

A **project**, and a **presentation of it to your classmates in week 13**. I will support
you throughout — that is what most of our time together is for.

The topic and specification are not fixed yet; we will settle them together in the next
few weeks, once you have enough of the machinery to judge what is a sensible scope. If you
already have a dataset or a domain you care about, start thinking about it now and come
talk to me.

### Homework

1. **Finish the exercise**, especially §3: your own rule from last week, as a query. Does it
   give back the number you submitted?
2. **Bring one business question.** About anything — sport, a job you have had, a hobby, this
   faculty. One sentence, written down. And be ready for the annoying version of it:

   > **who would do what differently, depending on the answer?**

If your environment did not work in class, run
[`exercises/00_hello_world.ipynb`](../exercises/00_hello_world.ipynb) and bring the output.

### If you want to read something

* Ralph Kimball, *[The Data Warehouse Toolkit](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/books/)* —
  the field's standard reference. I own a copy and will lend it out.
* The [dbt documentation](https://docs.getdbt.com/) — unusually well written.

---

*Questions? Now, after the session, or by email — all fine.*

## Appendix · The vocabulary

You will hear all of these all semester. Skim now, come back whenever a word is unfamiliar.

| term | what it means |
|---|---|
| **source system** | where data is born — the e-shop, the CRM, a sensor, an API. Also called *operational* or *OLTP* systems. |
| **OLTP** | *online transaction processing*: many tiny reads and writes, optimised for speed and safety. Your e-shop. |
| **OLAP** | *online analytical processing*: few enormous reads, optimised for aggregation. Your warehouse. |
| **ETL / ELT** | *extract, transform, load* — moving data from sources into the warehouse. ELT does the transform last, inside the warehouse, which is how it is usually done now. |
| **staging** | the first landing area: source data, renamed and typed, but not yet interpreted. |
| **data warehouse** | the central analytical store. Integrated, historical, organised for questions rather than transactions. |
| **fact** | a measurement — an amount, a quantity, a duration. Facts are what you `sum()`. |
| **dimension** | the context of a measurement — who, what, when, where. Dimensions are what you group *by*. |
| **grain** | what exactly one row of a fact table represents. The most important sentence in any data model. |
| **star schema** | one fact table surrounded by its dimensions. The standard shape. |
| **surrogate key** | a meaningless integer the warehouse assigns, instead of reusing the source system's ID. |
| **data mart** | a slice of the warehouse shaped for one department or one subject. |
| **semantic layer** | where metric definitions live, so that "revenue" means one thing everywhere. |
| **metric / KPI** | a defined, agreed number. A KPI is a metric somebody is accountable for. |
| **lineage** | the record of where a number came from, and what it passed through. |
| **SCD** | *slowly changing dimension* — how you keep history when a customer changes segment. |